In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_df = spark.read.table("bronze_iot_data")

silver_path = "s3://my-iot-data-bucket-2025/silver"

bronze_df_filted = bronze_df.withColumn("battery", col("battery").cast("int")) \
    .withColumn("humidity", col("humidity").cast("double")) \
    .withColumn("temperature", col("temperature").cast("double")) \
    .withColumn("timestamp", col("timestamp").cast("timestamp")) \
    .withColumn("event_date", date_format(col("timestamp"), "yyyy-MM-dd")) \
    .filter(col("device_id").isNotNull() & col("timestamp").isNotNull()) \
    .filter((col("temperature") > -50) & (col("temperature") < 70)).drop("_rescued_data") \
    .dropDuplicates(["device_id", "timestamp"])

is_faulty_df = bronze_df_filted.withColumn("is_faulty", when((col("battery") < 0) | (col("battery") > 100) | (col("humidity") < 0) | (col("humidity") > 100) | (col("temperature") < -20) | (col("temperature") > 60), True).otherwise(False))

final_df = is_faulty_df.withColumn("device_id", lower(col("device_id")))

final_df.write.format("delta").mode("overwrite").partitionBy("event_date").save(silver_path + "/iot")
# (final_df.writeStream
#     .format("delta")
#     .option("checkpointLocation", silver_path + "/silver/iot")
#     .outputMode("append")
#     .partitionBy("event_date")
#     .table("silver_iot_data"))